# 05 — Inflation Regime Detection

Statistical identification of inflation regimes in Spain CPI (2021–2024): stability (pre-shock), ECB shock, and normalization. Validates the three-period segmentation used in rolling-origin backtesting and Diebold-Mariano sub-period tests.

**Input:** `data/processed/ipc_spain_index.parquet`  
**Establishes:** Regime boundaries, rolling volatility, YoY acceleration phases

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT     = Path("..").resolve()   # tfg-forecasting/
MONOREPO = ROOT.parent            # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))

from shared.logger import get_logger

logger = get_logger(__name__)

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

## 1. Data Loading

In [ ]:
df = pd.read_parquet(ROOT / "data" / "processed" / "ipc_spain_index.parquet")
y  = df["indice_general"]
y.index = pd.to_datetime(y.index)

# Year-on-year change (%) — primary series for regime analysis
yoy = y.pct_change(12) * 100

# Test window: 2021-01 to 2024-12
test = yoy.loc["2021-01-01":"2024-12-31"]

print(f"Full series:  {y.index.min().date()} -> {y.index.max().date()}")
print(f"Test window:  {test.index.min().date()} -> {test.index.max().date()}")
print(f"Test obs:     {len(test)}")

## 2. Rolling Statistics — Detecting Regime Shifts

In [ ]:
# 12-month rolling mean and std of YoY change
roll_mean = yoy.rolling(12).mean()
roll_std  = yoy.rolling(12).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel A: YoY with rolling mean and regime shading
ax = axes[0]
ax.plot(yoy.index, yoy, color="#888", lw=0.8, alpha=0.6, label="YoY change")
ax.plot(roll_mean.index, roll_mean, color="#d62728", lw=2, label="12m rolling mean")
ax.axhline(0, color="black", lw=0.5)
ax.axhline(2, color="grey", lw=0.8, ls=":", label="ECB target (2%)")
ax.axvspan("2021-01-01", "2021-12-31", color="#d4e8d4", alpha=0.4, label="A: Stability")
ax.axvspan("2022-01-01", "2022-12-31", color="#f8d7d7", alpha=0.5, label="B: ECB Shock")
ax.axvspan("2023-01-01", "2024-12-31", color="#d7e8f8", alpha=0.35, label="C: Normalization")
ax.set_ylabel("YoY Change (%)")
ax.set_title("Spain CPI — Year-on-Year Change with Rolling Mean", fontweight="bold")
ax.legend(fontsize=8, loc="upper left", ncol=3)

# Panel B: Rolling volatility
ax2 = axes[1]
ax2.plot(roll_std.index, roll_std, color="#ff7f0e", lw=1.8, label="12m rolling std")
ax2.axvspan("2021-01-01", "2021-12-31", color="#d4e8d4", alpha=0.4)
ax2.axvspan("2022-01-01", "2022-12-31", color="#f8d7d7", alpha=0.5)
ax2.axvspan("2023-01-01", "2024-12-31", color="#d7e8f8", alpha=0.35)
ax2.set_ylabel("Rolling Std (pp)")
ax2.set_title("YoY Volatility (12-month rolling std)", fontweight="bold")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Regime Statistics — Mean, Volatility, Range

In [ ]:
REGIMES = {
    "A: Stability (2021)":        ("2021-01-01", "2021-12-31"),
    "B: ECB Shock (2022)":        ("2022-01-01", "2022-12-31"),
    "C: Normalization (2023-24)": ("2023-01-01", "2024-12-31"),
    "Global (2021-24)":           ("2021-01-01", "2024-12-31"),
}

print(f"{'Regime':<32} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'N':>5}")
print("-" * 72)
for name, (s, e) in REGIMES.items():
    seg = yoy.loc[s:e].dropna()
    print(f"{name:<32} {seg.mean():>8.2f} {seg.std():>8.2f} {seg.min():>8.2f} {seg.max():>8.2f} {len(seg):>5}")

## 4. Acceleration Analysis

Month-over-month change in the YoY rate (second derivative of the price level). Inflection points mark the onset of regime transitions.

In [ ]:
# Month-over-month change in YoY rate (acceleration)
accel = yoy.diff()
win   = accel.loc["2020-01-01":]

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["#d62728" if v > 0 else "#2ca02c" for v in win.values]
ax.bar(win.index, win.values, color=colors, alpha=0.75, width=25)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(pd.Timestamp("2022-01-01"), color="#cc0000", lw=1.5, ls="--", label="Shock start (2022-01)")
ax.axvline(pd.Timestamp("2023-01-01"), color="#0066cc", lw=1.5, ls="--", label="Normalization start (2023-01)")
ax.set_ylabel("Acceleration (pp/month)")
ax.set_title("Spain CPI — YoY Acceleration  |  Red = accelerating, Green = decelerating", fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 5. Full-Series Regime Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(yoy.index, yoy, color="#333", lw=1.5)
ax.fill_between(yoy.index, yoy, 0, where=(yoy > 0), alpha=0.18, color="#d62728")
ax.fill_between(yoy.index, yoy, 0, where=(yoy < 0), alpha=0.18, color="#2ca02c")

ax.axvspan("2021-01-01", "2021-12-31", color="#d4e8d4", alpha=0.45, zorder=0)
ax.axvspan("2022-01-01", "2022-12-31", color="#f8d7d7", alpha=0.50, zorder=0)
ax.axvspan("2023-01-01", "2024-12-31", color="#d7e8f8", alpha=0.35, zorder=0)

for lbl, x in [
    ("A: Stability",    "2021-02-01"),
    ("B: ECB Shock",    "2022-02-01"),
    ("C: Normalization","2023-02-01"),
]:
    ax.text(pd.Timestamp(x), yoy.max() * 0.90, lbl,
            fontsize=9, fontweight="bold", color="#333")

ax.axhline(0, color="black", lw=0.5)
ax.axhline(2, color="grey", lw=0.8, ls=":", alpha=0.6)
ax.set_ylabel("Year-on-Year Change (%)")
ax.set_title("Spain CPI — Inflation Regimes (2002–2024)", fontweight="bold")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
print("=" * 65)
print("INFLATION REGIME DETECTION — Spain CPI (test window 2021-2024)")
print("=" * 65)
print()
for name, (s, e) in REGIMES.items():
    seg = yoy.loc[s:e].dropna()
    print(f"  {name}")
    print(f"    Period: {s} -> {e}  ({len(seg)} months)")
    print(f"    Mean YoY: {seg.mean():.2f}%  |  Max: {seg.max():.2f}%  |  Std: {seg.std():.2f}pp")
    print()
print("Regime boundaries used in DM sub-period tests:")
print("  A: 2021-01 - 2021-12  (stability, low inflation, ECB hold)")
print("  B: 2022-01 - 2022-12  (supply/energy shock, ECB rate hikes begin)")
print("  C: 2023-01 - 2024-12  (YoY deceleration, monetary tightening transmitted)")